Extract POS and POS daily from newly files (Oct 2025) and old `pos.xlsx` file



In [1]:
%cd ../../
%load_ext dotenv
%dotenv

/Users/hoangle/Projects/untangling-people/ylva/fwo_models


In [2]:
from pathlib import Path

import pandas as pd
import polars as pl

# Load dims

In [3]:
path = "data/processed/dim_meals_1029.parquet"

dim_meals = pl.read_parquet(path)
dim_meals.head()

id,meal_codes,names,restaurants,meal_type,schoolyear,attributes,co2,src
i64,list[i64],list[str],list[i64],i64,str,list[str],f32,list[str]
0,[90000000],"[""Kikhernetaginea& syysomenajogurttia""]",[1],5,"""23-24""",[],0.41,"[""pos_Jan23-Oct24""]"
1,[90000001],"[""Rapea Meiramikana""]","[1, 4]",3,"""23-24""",[],1.26,"[""pos_Jan23-Oct24""]"
2,[7203],"[""Kasvismuhennos Caponata""]","[1, 2, 4]",4,"""23-24""",[],0.42,"[""pos_Jan23-Oct24"", ""menus_meals""]"
3,[9058],"[""TexMex-siemenpyöryköitä ja Arrabiattakastiketta"", ""TexMex-siemenpyöryköitä ja Arrabiattakastiket""]","[1, 2, 4]",4,"""24-25""","[""gluten_free"", ""vegan-kpl""]",0.56,"[""pos_Jan23-Oct24"", ""menus_meals"", … ""menus_week1-6""]"
4,[6877],"[""Kasvisjalapenonuggetteja, tomaattisalsaa"", ""Kasvis-jalapnuget ja tomatsals""]","[1, 2, 4]",4,"""24-25""",[],0.44,"[""pos_Jan23-Oct24"", ""menus_meals"", ""pos_Nov24-Mar25""]"


In [4]:
path = "data/processed/dim_restaurants.xlsx"

dim_restaurants = pl.read_excel(path)
dim_restaurants.head()

restaurant_id,restaurant,restaurant_short
i64,str,str
1,"""600 Chemicum""","""che"""
2,"""620 Exactum""","""exa"""
3,"""610 Physicum""","""phy"""
4,"""570 Viikuna""","""vik"""


# Extract POS and POS daily

In [5]:
PATH_DIR_POS_PROCESSED = Path("data/processed/pos/historical")
PATH_DIR_POS_DAILY_PROCESSED = Path("data/processed/pos_daily/historical")

### Pre-process dim tables

In [6]:
dim_meals = (
    dim_meals
    .select(
        pl.col('id').alias('meal_id'),
        pl.col('meal_codes').alias('meal_code')
    )
    .explode('meal_code')
)

dim_meals.head()

meal_id,meal_code
i64,i64
0,90000000
1,90000001
2,7203
3,9058
4,6877


In [7]:
dim_restaurants = dim_restaurants.drop('restaurant_short')

dim_restaurants.head()

restaurant_id,restaurant
i64,str
1,"""600 Chemicum"""
2,"""620 Exactum"""
3,"""610 Physicum"""
4,"""570 Viikuna"""


## Extract from new files (Oct 2025)

In [8]:
schema = {
    'date': pl.Date,
    'meal': pl.String,
    'meal_type': pl.String,
    'meal_code': pl.Int64,
    'restaurant': pl.String,
    'co2': pl.Float64,
    'pcs': pl.Int64,
    'src': pl.String,
}

pos_intermediate = pl.read_excel("data/inter/intermediate/*.xlsx", schema_overrides=schema)

pos_intermediate.head()

date,meal,meal_type,meal_code,restaurant,co2,pcs,src
date,str,str,i64,str,f64,i64,str
2025-09-01,"""Vegaani, Take away""",null,10093,"""570 Viikuna""",null,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,"""Bar Myöhä Bbq-seitanbowl""",null,200006,"""570 Viikuna""",null,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,"""Take away ruoka""",null,3215,"""570 Viikuna""",null,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,"""Kasvisjalapenonugetteja ja tom…",null,9043,"""570 Viikuna""",null,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,"""Pizza, Kasvis""",null,1009,"""570 Viikuna""",null,2,"""Data Viikuna 9-2025.csv"""


In [13]:
pos = (
    pos_intermediate
    
    .join(dim_meals, on='meal_code', how='left')
    .join(dim_restaurants, on='restaurant', how='left')

    .select('date', 'meal_id', 'restaurant_id', 'pcs', 'src')
)


pos.head()

date,meal_id,restaurant_id,pcs,src
date,i64,i64,i64,str
2025-09-01,141,4,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,276,4,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,214,4,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,39,4,1,"""Data Viikuna 9-2025.csv"""
2025-09-01,266,4,2,"""Data Viikuna 9-2025.csv"""


In [22]:
pos_daily = (
    pos
    .group_by('date', 'restaurant_id')
    .agg(
        pl.col('pcs').sum(),
        pl.concat_list('src').flatten()
    )
    .with_columns(
        pl.col('src').list.unique().list.join('|')
    )
    .sort('date')
)

pos_daily.head()

date,restaurant_id,pcs,src
date,i64,i64,str
2025-09-01,3,120,"""Sold lunches Kumpula 9-2025.cs…"
2025-09-01,1,901,"""Sold lunches Kumpula 9-2025.cs…"
2025-09-01,4,294,"""Data Viikuna 9-2025.csv"""
2025-09-01,2,238,"""Sold lunches Kumpula 9-2025.cs…"
2025-09-02,2,368,"""Sold lunches Kumpula 9-2025.cs…"


In [15]:
path = PATH_DIR_POS_PROCESSED / "new_file_Oct2025.xlsx"
path.parent.mkdir(exist_ok=True, parents=True)

pos.write_excel(path)

In [23]:
path = PATH_DIR_POS_DAILY_PROCESSED / "new_file_Oct2025.xlsx"
path.parent.mkdir(exist_ok=True, parents=True)

pos_daily.write_excel(path)

## Extract from old `pos.xlsx`

In [16]:
path = "data/processed/pos.xlsx"

pos_old = pl.read_excel(path)
pos_old.head()

id,restaurant,meal_id,datetime,pcs,src
i64,i64,i64,datetime[ms],i64,str
0,1,55,2023-01-02 10:31:00,1,"""Sold lunches"""
1,1,8,2023-01-02 10:32:00,1,"""Sold lunches"""
2,1,55,2023-01-02 10:32:00,1,"""Sold lunches"""
3,1,8,2023-01-02 10:35:00,1,"""Sold lunches"""
4,1,55,2023-01-02 10:36:00,2,"""Sold lunches"""


In [26]:
pos = (
    pos_old
    .with_columns(pl.col('datetime').dt.date().alias('date'))
    .rename({'restaurant': 'restaurant_id'})
    .group_by('date', 'meal_id', 'restaurant_id')
    .agg(
        pl.col('pcs').sum(),
        pl.concat_list('src').flatten()
    )
    .with_columns(
        pl.col('src').list.unique().list.join('|')
    )
    .sort('date')
)

pos.head()

date,meal_id,restaurant_id,pcs,src
date,i64,i64,i64,str
2023-01-02,161,1,6,"""Sold lunches"""
2023-01-02,212,1,1,"""Sold lunches"""
2023-01-02,8,1,78,"""Sold lunches"""
2023-01-02,53,1,14,"""Sold lunches"""
2023-01-02,55,1,165,"""Sold lunches"""


In [27]:
pos_daily = (
    pos
    .group_by('date', 'restaurant_id')
    .agg(
        pl.col('pcs').sum(),
        pl.concat_list('src').flatten()
    )
    .with_columns(
        pl.col('src').list.unique().list.join('|')
    )
    .sort('date')
)

pos_daily.head()

date,restaurant_id,pcs,src
date,i64,i64,str
2023-01-02,1,355,"""Sold lunches"""
2023-01-03,1,400,"""Sold lunches"""
2023-01-04,1,433,"""Sold lunches"""
2023-01-05,1,601,"""Sold lunches"""
2023-01-09,3,33,"""Sold lunches"""


In [28]:
path = PATH_DIR_POS_PROCESSED / "old_pos.xlsx"
path.parent.mkdir(exist_ok=True, parents=True)

pos.write_excel(path)

In [29]:
path = PATH_DIR_POS_DAILY_PROCESSED / "old_pos.xlsx"
path.parent.mkdir(exist_ok=True, parents=True)

pos_daily.write_excel(path)